In [1]:
%matplotlib inline

import gc
import sys
import rasterio
import geopandas as gpd
import datacube
import numpy as np
import xarray as xr
import rioxarray as rxr
import cartopy
from rasterio.features import rasterize
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import contextily as ctx

from datacube.utils.cog import write_cog
from matplotlib import colors as mcolours

sys.path.insert(1, "../Tools/")
from dea_tools.plotting import display_map
from dea_tools.landcover import plot_land_cover
from dea_tools.dask import create_local_dask_cluster
from dea_tools.dask import create_dask_gateway_cluster

In [2]:
client = create_local_dask_cluster(return_client=True)
#client = create_dask_gateway_cluster(profile='r5_4XL', workers=8)

/env/lib/python3.10/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 39419 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/jenna.guffogg@ga.gov.au/proxy/39419/status,
Dashboard: /user/jenna.guffogg@ga.gov.au/proxy/39419/status,Workers: 1
Total threads: 15,Total memory: 117.21 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:33481,Workers: 1
Dashboard: /user/jenna.guffogg@ga.gov.au/proxy/39419/status,Total threads: 15
Started: Just now,Total memory: 117.21 GiB
Comm: tcp://127.0.0.1:40249,Total threads: 15
Dashboard: /user/jenna.guffogg@ga.gov.au/proxy/41863/status,Memory: 117.21 GiB
Nanny: tcp://127.0.0.1:43287,


In [3]:
dc = datacube.Datacube(app='lc_images_for_animation')

In [4]:
#bounds: (112.85, -43.7, 153.69, -9.86)

lat_range = (-9.86, -43.7)
lon_range = (112.85, 153.69)
time = ('1988','2023')

In [6]:
query = {
    'time':time
}

lazy_ds = dc.load(product='ga_ls_landcover_class_cyear_3',
                 measurements='level4',
                 output_crs='EPSG:3577',
                dask_chunks={},
                resolution=(-1000, 1000),
                 **query)

In [8]:
lazy_ds.chunksizes

Frozen({'time': (1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1), 'y': (3840,), 'x': (4128,)})

In [14]:
lazy_ds.level4[0, ::3, ::3]

<xarray.DataArray 'level4' (y: 1280, x: 1376)> Size: 2MB
dask.array<getitem, shape=(1280, 1376), dtype=uint8, chunksize=(1280, 1376), chunktype=numpy.ndarray>
Coordinates:
    time         datetime64[ns] 8B 1988-07-01T23:59:59.999999
  * y            (y) float64 10kB -1.056e+06 -1.06e+06 ... -4.89e+06 -4.894e+06
  * x            (x) float64 11kB -1.92e+06 -1.916e+06 ... 2.202e+06 2.206e+06
    spatial_ref  int32 4B 3577
Attributes:
    units:         1
    nodata:        255
    crs:           EPSG:3577
    grid_mapping:  spatial_ref